# 5b — Aproximácia funkcie (MLP, PyTorch)

Spustiteľné zadanie: regresia na `datafun.csv` / `datafunindx.csv`, TensorBoard. Tento skript používa **tenzory priamo** (bez `DataLoader`) kvôli jednoduchej slučke a voľbe LBFGS/Adam — pozri poznámku v [`5_uvod_do_pytorch.ipynb`](5_uvod_do_pytorch.ipynb). **Teória:** [`5_uvod_do_pytorch.ipynb`](5_uvod_do_pytorch.ipynb).


In [ ]:
# ============================================================
# 5b - Aproximácia funkcie pomocou MLP v PyTorch
# ============================================================

# Ak treba:
# !pip install tensorboard pandas matplotlib

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter


# ============================================================
# 1. Seed a device
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)


# ============================================================
# 2. Načítanie dát
# ============================================================

DATAFUN_PATH = "datafun.csv"
DATAFUNINDX_PATH = "datafunindx.csv"

DATAFUN_URL = "https://raw.githubusercontent.com/STU-FEI-OUI/UMINT-UNS_data/main/Python_(CSV)/datafun.csv"
DATAFUNINDX_URL = "https://raw.githubusercontent.com/STU-FEI-OUI/UMINT-UNS_data/main/Python_(CSV)/datafunindx.csv"


def load_csv(local_path, remote_url):
    if os.path.exists(local_path):
        return pd.read_csv(local_path, header=None)
    return pd.read_csv(remote_url, header=None)


df_fun = load_csv(DATAFUN_PATH, DATAFUN_URL)
df_indx = load_csv(DATAFUNINDX_PATH, DATAFUNINDX_URL)

# datafun.csv: stĺpec 0 = x, stĺpec 1 = y
# ak by bol v súbore iba 1 stĺpec, toto by spadlo - čo je správne
x_all_orig = df_fun[[0]].values.astype(np.float32)
y_all_orig = df_fun[[1]].values.astype(np.float32)

# datafunindx.csv:
# stĺpec 0 = 1-based index
# stĺpec 1 = label (1=train, 2=test)
train_idx = (df_indx[df_indx[1] == 1][0].values.astype(int) - 1)
test_idx = (df_indx[df_indx[1] == 2][0].values.astype(int) - 1)

print("all samples :", len(x_all_orig))
print("train samples:", len(train_idx))
print("test samples :", len(test_idx))


# ============================================================
# 3. Train-only normalizácia
# ============================================================

x_train_orig = x_all_orig[train_idx]
y_train_orig = y_all_orig[train_idx]
x_test_orig = x_all_orig[test_idx]
y_test_orig = y_all_orig[test_idx]

x_mean = x_train_orig.mean(axis=0)
x_std = x_train_orig.std(axis=0) + 1e-8
y_mean = y_train_orig.mean(axis=0)
y_std = y_train_orig.std(axis=0) + 1e-8

x_all = (x_all_orig - x_mean) / x_std
y_all = (y_all_orig - y_mean) / y_std

x_train = x_all[train_idx]
y_train = y_all[train_idx]
x_test = x_all[test_idx]
y_test = y_all[test_idx]

print("scaled x train mean:", float(x_train.mean()))
print("scaled x train std :", float(x_train.std()))
print("scaled y train mean:", float(y_train.mean()))
print("scaled y train std :", float(y_train.std()))


# ============================================================
# 4. Tensory
# ============================================================

Xt = torch.tensor(x_train, dtype=torch.float32, device=DEVICE)
Yt = torch.tensor(y_train, dtype=torch.float32, device=DEVICE)

Xtest = torch.tensor(x_test, dtype=torch.float32, device=DEVICE)
Ytest = torch.tensor(y_test, dtype=torch.float32, device=DEVICE)

Xall = torch.tensor(x_all, dtype=torch.float32, device=DEVICE)


# ============================================================
# 5. Parametre
# ============================================================

hidden_dim = 27

# default podľa vašej požiadavky
optimizer_name = "lbfgs"   # "lbfgs" alebo "adam"

optimizer_presets = {
    "adam": {
        "epochs": 4000,
        "lr": 0.01,
    },
    "lbfgs": {
        "epochs": 300,
        "lr": 1.0,
        "max_iter": 50,
    },
}

cfg = optimizer_presets[optimizer_name]
TARGET_LOSS = 1e-4
log_dir = f"runs/5b_{optimizer_name}"


# ============================================================
# 6. Model
# ============================================================

class SimpleMLP(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=27, output_dim=1):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = torch.tanh(self.fc1(x))
        x = self.fc2(x)
        return x


model = SimpleMLP(input_dim=1, hidden_dim=hidden_dim, output_dim=1).to(DEVICE)
criterion = nn.MSELoss()

if optimizer_name == "lbfgs":
    optimizer = torch.optim.LBFGS(
        model.parameters(),
        lr=cfg["lr"],
        max_iter=cfg["max_iter"]
    )
else:
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg["lr"]
    )


# ============================================================
# 7. Tréning
# ============================================================

writer = SummaryWriter(log_dir=log_dir)

train_loss_hist = []
test_loss_hist = []

for epoch in range(1, cfg["epochs"] + 1):

    if optimizer_name == "lbfgs":
        def closure():
            optimizer.zero_grad()
            pred = model(Xt)
            loss = criterion(pred, Yt)
            loss.backward()
            return loss

        optimizer.step(closure)

    else:
        optimizer.zero_grad()
        pred = model(Xt)
        loss = criterion(pred, Yt)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        train_loss = criterion(model(Xt), Yt).item()
        test_loss = criterion(model(Xtest), Ytest).item()

    train_loss_hist.append(train_loss)
    test_loss_hist.append(test_loss)

    writer.add_scalar("loss/train", train_loss, epoch)
    writer.add_scalar("loss/test", test_loss, epoch)

    if epoch == 1 or epoch % max(1, cfg["epochs"] // 5) == 0 or epoch == cfg["epochs"]:
        print(
            f"Epoch {epoch:4d}/{cfg['epochs']} | "
            f"train_loss={train_loss:.8e} | "
            f"test_loss={test_loss:.8e}"
        )

writer.close()


# ============================================================
# 8. TensorBoard
# ============================================================

# Jupyter / Colab:
# %load_ext tensorboard
# %tensorboard --logdir runs


# ============================================================
# 9. Graf train_loss vs test_loss
# ============================================================

plt.figure(figsize=(9, 4.5))
plt.semilogy(train_loss_hist, label="train_loss")
plt.semilogy(test_loss_hist, label="test_loss")
plt.axhline(TARGET_LOSS, color="black", linestyle="--", label=f"cieľ {TARGET_LOSS:.0e}")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title(f"5b - train_loss a test_loss ({optimizer_name})")
plt.grid(True)
plt.legend()
plt.show()


# ============================================================
# 10. Predikcia v pôvodneho škále 
# ============================================================

model.eval()
with torch.no_grad():
    y_all_pred_scaled = model(Xall).cpu().numpy()

y_all_pred_orig = y_all_pred_scaled * y_std + y_mean

y_train_pred_orig = y_all_pred_orig[train_idx]
y_test_pred_orig = y_all_pred_orig[test_idx]


# ============================================================
# 11. Metriky v pôvodnej škále
# ============================================================

def regression_metrics(y_true, y_pred):
    err = y_true - y_pred
    sse = float(np.sum(err ** 2))
    mse = float(np.mean(err ** 2))
    mae = float(np.mean(np.abs(err)))
    return sse, mse, mae

train_sse, train_mse, train_mae = regression_metrics(y_train_orig, y_train_pred_orig)
test_sse, test_mse, test_mae = regression_metrics(y_test_orig, y_test_pred_orig)

print(f"train: SSE={train_sse:.8e}, MSE={train_mse:.8e}, MAE={train_mae:.8e}")
print(f"test : SSE={test_sse:.8e}, MSE={test_mse:.8e}, MAE={test_mae:.8e}")


# ============================================================
# 12. Graf pôvodná funkcia vs MLP simulácia
# ============================================================

x_plot = x_all_orig.squeeze()
y_plot = y_all_orig.squeeze()
y_pred_plot = y_all_pred_orig.squeeze()

order = np.argsort(x_plot)
x_plot = x_plot[order]
y_plot = y_plot[order]
y_pred_plot = y_pred_plot[order]

mae_all = np.mean(np.abs(y_plot - y_pred_plot))
mse_all = np.mean((y_plot - y_pred_plot) ** 2)

plt.figure(figsize=(10, 5))
plt.plot(x_plot, y_plot, "k-", linewidth=2, label="Pôvodná funkcia")
plt.plot(
    x_plot,
    y_pred_plot,
    "r--",
    linewidth=2,
    label=f"MLP simulácia | MAE={mae_all:.6e}, MSE={mse_all:.6e}"
)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Porovnanie pôvodnej funkcie a simulácie pomocou MLP")
plt.grid(True)
plt.legend()
plt.show()